In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
PROJECT_DIR = "/content/drive/MyDrive/PrismHire"

In [4]:
master_df = pd.read_parquet(
    f"{PROJECT_DIR}/master_candidates.parquet"
)

In [5]:
master_df.shape

(100000, 35)

In [6]:
master_df.head()

,candidate_id,years_experience,current_title,current_industry,current_company_size,country,num_skills,avg_skill_proficiency,avg_skill_endorsements,avg_skill_duration,...,github_activity_score,connection_count,endorsements_received,notice_period_days,willing_to_relocate,preferred_work_mode,verified_email,verified_phone,linkedin_connected,avg_assessment_score
0,CAND_0000001,6.9,Backend Engineer,IT Services,10001+,Canada,17,2.235294,17.176471,24.823529,...,9.2,356,35,60,0,3,1,1,0,49.725
1,CAND_0000002,12.5,Operations Manager,IT Services,10001+,India,9,1.666667,7.777778,20.333333,...,0.0,179,3,60,0,4,0,0,0,0.000
2,CAND_0000003,1.1,Customer Support,IT Services,10001+,USA,6,1.500000,7.833333,17.666667,...,0.0,19,46,150,0,2,1,0,0,0.000
3,CAND_0000004,3.8,Marketing Manager,Paper Products,201-500,Australia,10,1.500000,8.000000,13.400000,...,0.0,485,22,120,1,3,1,1,1,0.000
4,CAND_0000005,11.0,Accountant,Manufacturing,1001-5000,India,6,1.666667,14.666667,24.500000,...,0.0,300,14,30,1,2,0,1,1,0.000


In [7]:
def minmax(series):
    return (
        series - series.min()
    ) / (
        series.max() - series.min()
    )

In [8]:
master_df["num_skills_n"] = minmax(
    master_df["num_skills"]
)

master_df["skill_prof_n"] = minmax(
    master_df["avg_skill_proficiency"]
)

master_df["assessment_n"] = minmax(
    master_df["avg_assessment_score"]
)

master_df["github_n"] = minmax(
    master_df["github_activity_score"]
)

In [9]:
master_df["technical_dna"] = (
      0.30 * master_df["num_skills_n"]
    + 0.25 * master_df["skill_prof_n"]
    + 0.30 * master_df["assessment_n"]
    + 0.15 * master_df["github_n"]
)

In [10]:
master_df["exp_n"] = minmax(
    master_df["years_experience"]
)

master_df["job_duration_n"] = minmax(
    master_df["avg_job_duration"]
)

master_df["max_job_n"] = minmax(
    master_df["max_job_duration"]
)

master_df["industry_exp_n"] = minmax(
    master_df["industry_exposure"]
)

In [11]:
master_df["career_dna"] = (
      0.35 * master_df["exp_n"]
    + 0.30 * master_df["job_duration_n"]
    + 0.20 * master_df["max_job_n"]
    + 0.15 * master_df["industry_exp_n"]
)

In [12]:
master_df["leadership_n"] = minmax(
    master_df["leadership_experience"]
)

master_df["connections_n"] = minmax(
    master_df["connection_count"]
)

master_df["endorsements_n"] = minmax(
    master_df["endorsements_received"]
)

In [13]:
master_df["leadership_dna"] = (
      0.40 * master_df["leadership_n"]
    + 0.35 * master_df["connections_n"]
    + 0.25 * master_df["endorsements_n"]
)

In [14]:
master_df["notice_inverse"] = (
    180 - master_df["notice_period_days"]
)

In [15]:
master_df["notice_n"] = minmax(
    master_df["notice_inverse"]
)

master_df["response_n"] = minmax(
    master_df["recruiter_response_rate"]
)

master_df["interview_n"] = minmax(
    master_df["interview_completion_rate"]
)

master_df["saved_n"] = minmax(
    master_df["saved_by_recruiters"]
)

In [16]:
master_df["recruitability_dna"] = (
      0.30 * master_df["open_to_work"]
    + 0.20 * master_df["response_n"]
    + 0.20 * master_df["interview_n"]
    + 0.10 * master_df["saved_n"]
    + 0.20 * master_df["notice_n"]
)

In [17]:
master_df["search_n"] = minmax(
    master_df["search_appearances"]
)

master_df["views_n"] = minmax(
    master_df["profile_views_30d"]
)

master_df["profile_n"] = minmax(
    master_df["profile_completeness"]
)

In [18]:
master_df["market_dna"] = (
      0.40 * master_df["search_n"]
    + 0.30 * master_df["views_n"]
    + 0.30 * master_df["profile_n"]
)

In [19]:
master_df[
    [
        "technical_dna",
        "career_dna",
        "leadership_dna",
        "recruitability_dna",
        "market_dna"
    ]
].describe()

,technical_dna,career_dna,leadership_dna,recruitability_dna,market_dna
count,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000
mean,0.188260,0.267818,0.136428,0.379950,0.197249
std,0.124852,0.123989,0.064971,0.168894,0.083832
min,0.000000,0.016970,0.000000,0.014503,0.004152
25%,0.094444,0.174784,0.089732,0.247490,0.132502
50%,0.141667,0.269234,0.130598,0.336594,0.196168
75%,0.268262,0.362621,0.176049,0.530152,0.256685
max,0.830454,0.744287,0.578919,0.897997,0.865002


In [20]:
genome_columns = [
    "technical_dna",
    "career_dna",
    "leadership_dna",
    "recruitability_dna",
    "market_dna"
]

In [21]:
candidate_genome = master_df[
    genome_columns
].copy()

In [22]:
candidate_genome.shape

(100000, 5)

In [23]:
candidate_genome.insert(
    0,
    "candidate_id",
    master_df["candidate_id"]
)

In [24]:
candidate_genome.to_parquet(
    f"{PROJECT_DIR}/models/candidate_genome.parquet",
    index=False
)

In [25]:
candidate_genome.head()

,candidate_id,technical_dna,career_dna,leadership_dna,recruitability_dna,market_dna
0,CAND_0000001,0.476048,0.312601,0.100299,0.610960,0.333225
1,CAND_0000002,0.122222,0.452463,0.263000,0.581993,0.249427
2,CAND_0000003,0.058333,0.023413,0.049189,0.259624,0.035956
3,CAND_0000004,0.125000,0.154767,0.225069,0.115899,0.017767
4,CAND_0000005,0.072222,0.401792,0.182509,0.662233,0.266331


In [26]:
candidate_genome.describe()

,technical_dna,career_dna,leadership_dna,recruitability_dna,market_dna
count,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000
mean,0.188260,0.267818,0.136428,0.379950,0.197249
std,0.124852,0.123989,0.064971,0.168894,0.083832
min,0.000000,0.016970,0.000000,0.014503,0.004152
25%,0.094444,0.174784,0.089732,0.247490,0.132502
50%,0.141667,0.269234,0.130598,0.336594,0.196168
75%,0.268262,0.362621,0.176049,0.530152,0.256685
max,0.830454,0.744287,0.578919,0.897997,0.865002
